# Processing solar corona images from total eclipse

This notebook shows full solar eclipse image processing pipeline. Current version works with Nicole Sharp's [data](https://www.cloudynights.com/forums/topic/919153-have-my-eclipse-raws/#findComment-13397853).

First few cells with auxiliar uninteresting stuff

In [ ]:
import sys
from pathlib import Path


def v0_package_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "eclipse_v1").is_dir():
        return cwd
    if (cwd / "v1" / "eclipse_v1").is_dir():
        return cwd / "v1"
    raise RuntimeError(
        "Could not find eclipse_v1: set cwd to the `v1` directory or the repo root."
    )


V0_ROOT = v0_package_root()
if str(V0_ROOT) not in sys.path:
    sys.path.insert(0, str(V0_ROOT))

In [ ]:
# GPU selection — same defaults as legacy `eda00.py`. Override in the shell, e.g.
#   CUDA_VISIBLE_DEVICES=0 jupyter lab
from eclipse_v1.device import configure_cuda_visible_devices, require_cuda

configure_cuda_visible_devices()
require_cuda()

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("EDA00_DATA_ROOT", "/home/slavik/e202602_eclipse/data"))
WORKDIR = Path(os.environ.get("ECLIPSE_V1_WORKDIR", "/home/slavik/tmp/eclipse_v1_run"))
WORKDIR.mkdir(parents=True, exist_ok=True)

PK_EDA00 = WORKDIR / "v1-eda00.pkl"
PK_EDA02 = WORKDIR / "v1-eda02.pkl"
PK_EDA03 = WORKDIR / "v1-eda03.pkl"

## Stage 0 — ingest, moon, intra-exposure registration

### 0a — Scan disk → `ImageInfo` list

In [ ]:
import pipeline
from eclipse_v1 import stage0 as s0

# find all images, obtain basic informations about them (size, avg brighness, etc)
image_infos = s0.get_image_infos(DATA_ROOT)
print(f'We have {len(image_infos)} images, first one is {image_infos[0].path}')
# group images by exposure time
exposure_groups = s0.stage0_group_by_exposure(image_infos)
print('\nExample images - one per each exposure time')
pipeline.plot_exposure_group_thumbnails(exposure_groups)

### 0b — Moon detection (loop over every frame)

Our Moon detection is based on assumption that Moon is dark circle surrounded with brighter cloud, where the cloud has sharp circular inner border. One difficulty which makes it nontrivial are protuberances - small blobs much brighter than rest of the corona. They can confuse many naive approaches. And the assumption "sharp circular inner border" is not valid for long exposures where light polute area of Moon disc. Our algorithm fails there, we just detect this failure, erase affected Moon info and replace it with interpolation from successful frames.

Our algorithm is overcomplicated, I don't claim hat it has to be so complex. Anyway, here it is: `stage0_detect_moons()` first downscale image to manageable size, then examine all possible pairs of concentric circles where bigger circle radius is 2px bigger than smaller one, for each circle it compute sum of pixel values around circle boundary, and it pick the circle pair with biggest difference of this quantity between bigger and smaller circle. This gives first approximate of Moon center. With this in hand we move to full resolution, where we split image to 360 radial 1° pie segments from this approximated center. In each segment we identify point with biggest radial gradient. So we have 360 candidate border points. From these points we form 1024 random triplets and for each triplet we find circumcenter. So we have cluster 1024 center candidates. Using AHC library we find dense core of this cluster defined 256 most tightly packed points. We average them obtaining final Moon center. And because for each center candidate we have know the correspnding Moon radius (as we know the triplet of border points which gave us the center candidate), we can average these radii for our 256 best center candidates, obtaining Moon radius estimation.

Looking at the results: shorter exposure yields quite consistent Moon estimation (small `np.std(radii)` within the group). An Moon radious gently decrease with growing exposure time. Of course the Monn doesn't shrink physically, but for longer exposures the light starts to polute moon disk area due to camera imperefections. This phenomenon explode for really long exposures where Moon parameters are entirely off.

In [ ]:
# in each image independently, determine moon position and radius
s0.stage0_detect_moons(image_infos)
print(f'Exposure groups initial state')
s0.stage0_print_exposure_groups_stats(exposure_groups)
# Some moon estimations are just wrong. Identify them and erase them
print(f'Pruning failed Moon estimations')
s0.stage0_prune_moon_info_for_radius_outliers(exposure_groups)

### 0c — Interpolate missing positions; set position uncertainty

In [ ]:
interp = s0.stage0_interpolate_missing_moons(image_infos, exposure_groups)
s0.stage0_set_moon_position_std(image_infos, exposure_groups, interp)

### 0d — Intra-exposure registration: all ordered pairs per group (slow)

Slowest part of pipeline, registration (alignment) of images within each exposure group. It's slow because we are computing alignment of each image pair within group, distinguishing even order (A to B computed seprately from B to A). And the registration is nontrivial because we need to align corona (blurry), not the Moon border which is nice and sharp, but is significantly moving wrt our real target - the corona.

Our regisrtation algorithm does gridsearch of transformations of image B which would lead to good alignment with image A. For scoring 'goodness' of alignment we mask out Moon from both images, filter out lowfeq tangential components, clip some extremely high intensities (protuberances), and  use mean absolute error. Cliping of protuberance intensities is because protuberances evolve quickly. We clip the intensities because we still want to use protuberances as landmarks, but we dont want them to dominate the alignment score. 

Regarding output: for each exposure group you see progress of its processing and once it's done, there is output like
```
Check 1: ij mean=0.1423 max=0.5546  rot mean=0.0104 max=0.0781
Check 2: ij mean=0.2621 max=1.1061  rot mean=0.0339 max=0.0781
```
Its doing two checks. Check 1 takes all registration pairs of images `A`, `B` and checks how much is found alignment transformation `A->B` consistent with `B->A`. It outputs stats of found discrepancy in pixels, first for translation (in our example values `mean=0.1423 max=0.5546`) and then for rotation component of found transform (in our example `rot mean=0.0104 max=0.0781`). Check 2 does similar discrepance check between composed transform `A->B, B->C` and direct `A->C` for all triplets in exposure group.

In [ ]:
reg = s0.stage0_register_intra_exposure_pairs(exposure_groups)
s0.stage0_save_pickle(exposure_groups, reg, PK_EDA00)

## Stage 1 — prune stacks, global pose fit per exposure

### 1a — Pruning of images within exposure groups

It has two steps. First we throw away images with average brightness too different from rest of the group. Then we remove images, which contribute most to inconsistent registrations (with too big difference between composed `A->B, B->C` and direct `A->C`).


In [ ]:
import torch
from eclipse_v1 import stage1 as s1

exposure_groups, reg = s1.stage1_load(PK_EDA00)
s1.stage1_prune_groups(exposure_groups, reg)

### 1b — Optimization loop (one solve per exposure time)

So far we have pairwise registration transformations between image pairs within each group. We need to figure out some absolute placement of the images withing group, which will be as consistent as possible with these known pairwise deltas. It's what the `stage1_optimize_poses_and_debug` does (in each group separately). It first builds some initial guess, starting from closest image pair and adding other images iterativly, in each step choosing the closest image to already placed. Then it just optimize global placement, minimizing the discrepancy between pairwise transforms computed from the global placement and pairwise transforms required by optimal pair registrations. Apart from solving the placement, it save some debug images and animations in `${WORKDIR}/v1-eda02_debugimg*`

In [ ]:
device = torch.device("cuda")
opt_results = s1.stage1_optimize_poses_and_debug(
    exposure_groups, reg, device, debug_img_dir=WORKDIR
)

Inspect carefuly debug images. Click them to see full resolution. The images ar animations. You should see Sun-related features (protuberances, corona) static and Moon jumping in front of them. If the Sun is not aligned perfectly, then the processing pipeline failed for your data and you have to debug it. On images with longre exposures, Sun features are not so well localized, but you start to be able to see stars. Stars are good enough to judge registration.

In [ ]:
s1.symlink_eda02_debug_anim_gifs(WORKDIR)
s1.display_clickable_eda02_debug_img_grid(columns=8, width=128)

In [ ]:
s1.stage1_save_pickle(PK_EDA02, exposure_groups, reg, opt_results)

## Stage 2 — full-res stack mean per exposure, then cross-exposure chain

### 2a — Load pickled data, compute moon parameters median per exposure group

In [ ]:
from eclipse_v1 import stage2 as s2

exposure_groups, _reg, opt_results = s2.stage2_load(PK_EDA02)
moon_by_exp, exposure_times_sorted = s2.stage2_moon_median_table(exposure_groups)

### 2b — Full-resolution averaged image per exposure

Use the registration parameters we computed in previous steps and compute single averaged image per each exposure group.

In [ ]:
import torch
device = torch.device("cuda")
avg_images = s2.stage2_fullsize_averages(
    exposure_groups, exposure_times_sorted, opt_results, device
)

### 2c — Consecutive exposure pairs: gamma + cross registration

For each pair of consecutive exposures we compute two quantities needed to perform registration (alignment):

1) `gamma`, exponent used to transform pixel values between images with different exposures

2) registraction (alignment) transformation.

Once we know `gamma`, we can convert pixel values between the two images and the transformation then can be computed almost the same way as we did it in previous stages (only we have to avoid pixels which are saturated in the image with longer exposure). But to compute `gamma` (which we do by just gridsearch with MAE as objective), one needs to know how to align the two images. So we do 1) and 2) in two iterations, refining each of them using output of the another.

In [ ]:
pairs_results = s2.stage2_cross_exposure_consecutive_pairs(
    exposure_times_sorted, avg_images, moon_by_exp, device, pair_gif_dir=WORKDIR
)

Create debug gifs, similarly as we did in previous stage. Inspect, if registration between exposure pairs worked.

In [ ]:
s2.symlink_eda03_pair_gifs(WORKDIR)
s2.display_clickable_eda03_pair_gif_grid(columns=8, width=128)

Pickle results of this stage

In [ ]:

s2.stage2_save_pickle(PK_EDA03, pairs_results)

## Stage 3 — reference merge, radial tone, **patch FFT sharpen**, RGB

The next code cell uses a **`Stage3Context`** and explicit **`stage3_*` steps** (same math as `run_stage3(WORKDIR)`).

**A →** `stage3_load_inputs` — reload `v1-eda02.pkl` / `v1-eda03.pkl`; drop first two exposure times; set reference and `moon_ref`.

**B →** `stage3_build_per_exposure_averages` then `stage3_warp_merge_to_composite` — GPU stack means; chain `cross_reg` + gamma scaling; weighted merge to reference grid.

**C →** `stage3_crop_and_save_composite` — mutual-valid crop; `v1-eda05_composite.npy` + preview.

**D →** `stage3_radial_normalize_display` — moon refine, polar tone + **p3** stretch → `ctx.display`.

**E →** `stage3_fft_unsharp_and_save` — patch FFT unsharp (σ ∈ {2,4,8}, weights `STAGE3_UNSHARP_WEIGHTS`).

**F →** `stage3_rgb_vignette_and_radial_pickle` — RGB + vignette; `v1-eda05_radial.pkl`.

**A. Reload** stage 1+2 pickles; drop first two exposure times (same as `eda05`); pick **reference** = shortest remaining; recompute per-exposure averages in that time list.

**B. Warp + weighted merge** every exposure into the reference grid (chain `cross_reg`, gamma-based intensity scaling, radial weights).

**C. Crop** to mutual valid footprint; save float composite + preview PNG.

**D. Radial pipeline (GPU):** warp to polar, angular statistics / extrapolation, piecewise tone map, **p3** percentile curve, normalize — details in library; outcome is a display-range grayscale `display`.

**E. FFT sharpen (this is the patch loop you asked to surface):**
- Build **difference** `display - blurred(display)` with plain Gaussian and a polar-band blur (library).
- **Tile** the image with square patches (`STAGE3_PATCH_SIDE` × `STAGE3_PATCH_SIDE`, stride `STAGE3_PATCH_STRIDE`).
- **For each patch origin** `(r0, c0)` and each blur scale **σ ∈ {2,4,8}**:
  - `FFT2` → `fftshift` → **spectral processing** (amplitude shaping in polar layout + percentile gates; moon-aware masking — **hidden in library**) → `ifftshift` → `IFFT2` → real part.
  - **Overlap-add** with a smooth circular window.
- **Combine** the three smoothed residual maps with weights `STAGE3_UNSHARP_WEIGHTS`, add back to `display`, clip, zero moon disk.

**F. RGB + vignette** on sharpened gray; save PNG and `v1-eda05_radial.pkl` sidecar.

First we load what we saved from previous stages and use it to build sinlge image, which contains data from all the exposures. In previous stages we obtained registrations transforms and gammas which let us to convert pixel intensities fro mdifferent exposures to the same scale. So we do per-pixel weighted average of exposures. The average is 'weighted' because for each pixel some exposures are overburn, some are almost zero, and some provide somemeaningful reliable values.

In [ ]:
from eclipse_v1.stage3 import (
    Stage3Context,
    stage3_build_per_exposure_averages,
    stage3_crop_and_save_composite,
    stage3_fft_unsharp_and_save,
    stage3_load_inputs,
    stage3_radial_normalize_display,
    stage3_rgb_vignette_and_radial_pickle,
    stage3_warp_merge_to_composite,
    symlink_and_display_clickable,
)

ctx = Stage3Context(workdir=WORKDIR)
stage3_load_inputs(ctx)
# we build per exposure avgs again as we didnt saved them in previous stages
stage3_build_per_exposure_averages(ctx)
# single image from all the exposures
stage3_warp_merge_to_composite(ctx)
# crop the borders which are not covered enough
stage3_crop_and_save_composite(ctx)
symlink_and_display_clickable(ctx, 'v1-eda05_composite_preview.png')

The composite data already contains all the corona details through the whole image, but because huge dynamic range, they are not visible in image. Far from Sun they are too dimm, near the sun they are too bright. We have to somehow normalize them, to convert them into some perceivable values for all radius values. Simplest approach is to for each distance from sun center, compute mean and std of pixels at this distance, and use them to normalize pixel vaues at this distance. The actual formula I'm using is more elaborated but there is no science behind it. I just arrived by trial-error to something what looks acceptable for me. So the actual procedure is this:

To make implementation simpler, we work in polar coordinates with zero in Sun center.

For the bigger radii (corners of the image), we dont have pixels for all thetas. We need them to compute means around circumference, so we have simple intrapolation / extrapolation to compute them. So we can compute means. The actual normalization formula work as piecewise linear mapping `[mean/2, mean] -> [0, 0.4]` and `[mean, 2*mean] -> [0.4, 1]`. And in fact we compute two versions of it. In the first, mean is taken around whole circumference. In the second, mean is taken on sliding window 15% of circumference around the pixel we normalize. We average these two normalized values. Then we do one more step which Claude named 'radially-adaptive background subtraction'. The idea is to take some small percentile of pixels value as background intensity and subtract it, converting pixels with this (and lower) values to black. The percentile is radius dependent, varying linearly from 0.01% at sun center to 3% in image corners. 

In [ ]:
stage3_radial_normalize_display(ctx)
symlink_and_display_clickable(ctx, "v1-eda05_radial_normalize.png")

So here we finally see the corona but the image is blured. There is classical vintage algorithm for sharpening called Unsharp Mask. For image `x` it first makes its blurred copy `b = gaussian_blur(x, sigma)` and then it compute sharpened image as `y = x + c * (x - b)` where `c` is some scalar constant (strength). But it enhance noise as well as desired structure. To address this problem, I developed slightly elaborated version.

The bluring step is almost standard with one modification. Near the Moon boundary, bluring kernel is modified to prevent the moon boundary from geting blured radially. Once we have blured version, we compute the difference original minus blurred and then we try to enhance linear structures in this difference. To this end we process it in overlaping patches 256x256 pixels. In each patch we compute FFT. Then we use two facts: 1) the interesting structures in image tend to have big amplitudes in spectrum (or at least some of the amplitudes). 2) linear structures in image manifest in spectrum as radial lines. So, to exploit 1), we compute some amplitude threshold (specified by percentile) and we subtract the threshold from all amplitudes (and then clamp to prevent negative amplitudes). And before doing this, we exploit 2) to enhance the spectrum: we modify amplitudes, adding to each of them median along some line segment pointing towards spectrum plane center (which enhance radial lines in spectrum). 

This is main principle giving us enhanced difference wrt blurred version. We use this difference in standard unsharp mask way. Two more details: First, because we know where moon boundary (which is very sharp linear structure) is located in image, we can in each tile which contains it give its spectrum some special handling, keeping it separately from spectrum of corona features. Second, we are doing all this unsharp mask magic for 3 different values of sigma and strength, combining the results.

In [ ]:
stage3_fft_unsharp_and_save(ctx)
symlink_and_display_clickable(ctx, 'v1-eda05_radial_normalize_sharpen_fft_smoothed_diff.png')

The image is now much sharper but it comes at a expense: we introduced some artifacts. They are most notable at the doublestar which our trickery changed from this

![doublestar0](doublestar0.png)

to this

![doublestar1](doublestar1.png)

But the whole image with all the details is so beutiful, that I chose to accept the artifacts.

Finally, we apply some false colors and darkening towards border, to make it nicer

In [ ]:
stage3_rgb_vignette_and_radial_pickle(ctx)
symlink_and_display_clickable(ctx, 'v1-eda05_rgb_rescaled.png')

### After run: patch count on actual composite crop

## Quick look (optional)

---

**One-shot alternative** (same math, fewer visible steps): `s0.run_stage0(DATA_ROOT, PK_EDA00)`, `s1.run_stage1(...)`, `s2.run_stage2(...)`, and either `run_stage3(WORKDIR)` or the stepped `Stage3Context` + `stage3_*` calls above.